In [8]:
import nltk.corpus
from collections import Counter
from nltk.util import bigrams
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures

nltk.download('brown')
brown = nltk.corpus.brown
bwords = brown.words()

[nltk_data] Downloading package brown to
[nltk_data]     /Users/jadenvanrijswijk/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [15]:
def calculate_pmi(bwords, only_positive=False, freq_threshold=10):
    word_freq = Counter(bwords)
    bigram_freq = Counter(bigrams(bwords))

    # Filter out words that appear less than 10 times
    valid_words = {word for word, freq in word_freq.items() if freq >= freq_threshold}
    filtered_bigram_freq = {
        bigram: freq 
        for bigram, freq in bigram_freq.items() 
        if bigram[0] in valid_words and bigram[1] in valid_words
    }

    # PMI calculation
    bigram_finder = BigramCollocationFinder.from_words(bwords)
    bigram_finder.apply_word_filter(lambda w: word_freq[w] < freq_threshold)

    all_pmi_scores = {
        bigram: bigram_finder.score_ngram(BigramAssocMeasures.pmi, bigram[0], bigram[1])
        for bigram in filtered_bigram_freq
        if bigram_finder.score_ngram(BigramAssocMeasures.pmi, bigram[0], bigram[1]) is not None
    }
    
    if only_positive:
        all_pmi_scores = {bigram: max(pmi, 0) for bigram, pmi in all_pmi_scores.items()}

    sorted_pmi = sorted(all_pmi_scores.items(), key=lambda x: x[1], reverse=True)
        
    return sorted_pmi
        
# sorted_pmi = calculate_pmi(bwords, only_positive=False)
# print("=== Top 20 Bigrams by PMI ===")
# for bigram, pmi in sorted_pmi[:20]:
#     print(f"  {bigram}: {pmi:.4f}")

# print("\n=== Bottom 20 Bigrams by PMI ===")
# for bigram, pmi in sorted_pmi[-20:]:
#     print(f"  {bigram}: {pmi:.4f}")


The implementation of PMI on the Brown corpus has expected results. The highest scoring pair is ('Hong', 'Kong'), this is expected since it is a so called split word. The words have very little meaning on their own, only as a pair is when they make sense. Additionally the words have no other meaning in the English language except for the reference to the country Hong Kong. The high PMI is due to the low usage of the words not as a pair.

The lowest scoring word pairs on PMI consist mainly of stop words that are grammatically incorrect when the order is wrong. The lowest scoring pair is ('.', ','); which is coherent with the English grammar, just like ('the', 'I') and ('the', 'and'). 

The PMI on the corpus proves to be valid however, the results are biased to higher scores for split-words and lower from stop words. Further research should investigate the implementation with preprocessed data that remove these biases. 

In [18]:
def compare_pmi_ppmi(bwords):
    pmi_scores = calculate_pmi(bwords, only_positive=False, freq_threshold=1)
    ppmi_scores = calculate_pmi(bwords, only_positive=True, freq_threshold=1)

    print("\n=== Bottom 20 Bigrams by PMI and PPMI ===")
    for (bigram_pmi, pmi), (bigram_ppmi, ppmi) in zip(pmi_scores[-20:], ppmi_scores[-20:]):
        print(f"  {bigram_pmi}: PMI={pmi:.4f}, PPMI={ppmi:.4f}")
        
compare_pmi_ppmi(bwords[:100])
compare_pmi_ppmi(bwords)


=== Bottom 20 Bigrams by PMI and PPMI ===
  (',', '``'): PMI=4.0589, PPMI=4.0589
  ('election', 'was'): PMI=4.0589, PPMI=4.0589
  ('``', 'irregularities'): PMI=4.0589, PPMI=4.0589
  ('irregularities', "''"): PMI=4.0589, PPMI=4.0589
  ('primary', 'which'): PMI=4.0589, PPMI=4.0589
  ('which', 'was'): PMI=4.0589, PPMI=4.0589
  ('the', 'City'): PMI=3.8365, PPMI=3.8365
  ('deserves', 'the'): PMI=3.8365, PPMI=3.8365
  ('the', 'praise'): PMI=3.8365, PPMI=3.8365
  ('for', 'the'): PMI=3.8365, PPMI=3.8365
  ('the', 'manner'): PMI=3.8365, PPMI=3.8365
  ('the', 'hard-fought'): PMI=3.8365, PPMI=3.8365
  ('in', 'which'): PMI=3.4739, PPMI=3.4739
  ("''", 'in'): PMI=3.4739, PPMI=3.4739
  ('City', 'of'): PMI=3.3219, PPMI=3.3219
  ('the', 'election'): PMI=3.2515, PPMI=3.2515
  ('that', 'the'): PMI=2.8365, PPMI=2.8365
  ('of', 'the'): PMI=2.5146, PPMI=2.5146
  ('which', 'the'): PMI=2.2515, PPMI=2.2515
  ('in', 'the'): PMI=2.2515, PPMI=2.2515

=== Bottom 20 Bigrams by PMI and PPMI ===
  ('in', 'of'): PMI

PPMI shows no difference in scores on the brown100 corpus since all PMI scores are positive. However, on the full brown corpus, the lowest scoring bigrams have a PPMI of 0, which is expected since they have negative PMI scores. The highest scoring bigrams have the same PMI and PPMI scores since they are already positive, thus PPMI isn't a useful measure for this corpus since it loses information on the negative PMI scores. For example the bigram ('.', ',') has a PMI of -11.27 and a PPMI of 0, which is a significant loss of information.